# Preparation

In [1]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices[0])

PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [2]:
# === Phase 1: Global Config and Imports ===
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
import zipfile

# Configuration dictionary
CONFIG = {
    'SEED': 42,
    'INPUT_SIZE': 224,
    'BATCH_SIZE': 32,
    'TEST_SIZE': 0.2,
    'KFOLD': 3,
    'ZIP_PATH': 'train_images.zip',
    'IMAGE_DIR': 'unzipped/train_images',
    'CSV_PATH': 'meta_train.csv',
    'TRAIN_CSV': 'train.csv',
    'TEST_CSV': 'test.csv',
    'MODEL_DIR': 'saved_models/',
    'HISTORY_DIR': 'saved_histories/',
    'EPOCHS': 100,
    'EARLY_STOP_PATIENCE': 6,
    'LEARNING_RATE': 1e-3
}

# === Phase 1.5: Unzip Train Images ===
def unzip_train_images():
    if not os.path.exists('unzipped'):
        os.makedirs('unzipped')

    with zipfile.ZipFile(CONFIG['ZIP_PATH'], 'r') as zip_ref:
        zip_ref.extractall('unzipped')

    print(f"Unzipped {CONFIG['ZIP_PATH']} successfully.")

unzip_train_images()

# Set random seeds
random.seed(CONFIG['SEED'])
np.random.seed(CONFIG['SEED'])
tf.random.set_seed(CONFIG['SEED'])

# === Phase 2: Load Metadata ===
meta = pd.read_csv(CONFIG['CSV_PATH'])

# === Phase 3: Validate Images (missing or corrupt) ===
def validate_images(meta):
    missing = []
    corrupt = []
    for iid, label in zip(meta['image_id'], meta['label']):
        fname = iid if iid.lower().endswith('.jpg') else f"{iid}.jpg"
        fpath = os.path.join(CONFIG['IMAGE_DIR'], label, fname)
        if not os.path.isfile(fpath):
            missing.append(fpath)
        else:
            try:
                with Image.open(fpath) as img:
                    img.verify()
            except:
                corrupt.append(fpath)
    return missing, corrupt

missing_files, corrupt_files = validate_images(meta)
print(f"Missing files: {len(missing_files)}")
print(f"Corrupted files: {len(corrupt_files)}")

# === Phase 4: Summarize Image Resolutions BEFORE Rotation ===
def summarize_resolution(meta, title="Resolution Summary"):
    resolutions = []
    for iid, label in zip(meta['image_id'], meta['label']):
        fname = iid if iid.lower().endswith('.jpg') else f"{iid}.jpg"
        fpath = os.path.join(CONFIG['IMAGE_DIR'], label, fname)
        with Image.open(fpath) as img:
            w, h = img.size
            resolutions.append((w, h))
    df_res = pd.DataFrame(resolutions, columns=['width', 'height'])
    summary = df_res.groupby(['width', 'height']).size().reset_index(name='count')
    print(f"\n=== {title} ===")
    print(summary)
    return summary

summarize_resolution(meta, title="Resolution BEFORE Rotation")

# === Phase 5: Rotate Landscape Images ===
def rotate_landscape(meta):
    landscapes = []
    for iid, label in zip(meta['image_id'], meta['label']):
        fname = iid if iid.lower().endswith('.jpg') else f"{iid}.jpg"
        fpath = os.path.join(CONFIG['IMAGE_DIR'], label, fname)
        with Image.open(fpath) as img:
            w, h = img.size
            if w > h:
                landscapes.append(fpath)

    for path in landscapes:
        img = Image.open(path)
        img_rotated = img.transpose(Image.ROTATE_90)
        img_rotated.save(path)

    print(f"Rotated {len(landscapes)} landscape images.")

rotate_landscape(meta)

# === Phase 6: Summarize Image Resolutions AFTER Rotation ===
summarize_resolution(meta, title="Resolution AFTER Rotation")

# === Phase 7: Add Age Bins for Stratified Split ===
meta['age_bin'] = pd.qcut(meta['age'], q=5, labels=False, duplicates='drop')

# === Phase 8: Train/Test Split with Stratification ===
train_val, test = train_test_split(
    meta,
    test_size=CONFIG['TEST_SIZE'],
    stratify=meta['age_bin'],
    random_state=CONFIG['SEED']
)

# === Phase 9: Scale Age ===
age_mean = train_val['age'].mean()
age_std = train_val['age'].std()

train_val['age_scaled'] = (train_val['age'] - age_mean) / age_std
test['age_scaled'] = (test['age'] - age_mean) / age_std

# === Phase 10: Save Clean CSVs for tf.data Loading ===
def safe_image_path(row):
    fname = row['image_id'] if row['image_id'].lower().endswith('.jpg') else f"{row['image_id']}.jpg"
    return os.path.join(CONFIG['IMAGE_DIR'], row['label'], fname)

train_val['image_path'] = train_val.apply(safe_image_path, axis=1)
test['image_path'] = test.apply(safe_image_path, axis=1)

train_val[['image_path', 'age_scaled']].to_csv(CONFIG['TRAIN_CSV'], index=False)
test[['image_path', 'age_scaled']].to_csv(CONFIG['TEST_CSV'], index=False)

# === Phase 11: Compute RGB Mean and Std for Normalization ===
def compute_rgb_stats(image_paths):
    sum_rgb = np.zeros(3)
    sum_sq = np.zeros(3)
    total_pixels = 0

    for path in image_paths:
        img = np.array(Image.open(path).convert('RGB'), dtype=np.float32) / 255.0
        sum_rgb += img.sum(axis=(0, 1))
        sum_sq += (img ** 2).sum(axis=(0, 1))
        total_pixels += img.shape[0] * img.shape[1]

    mean_rgb = sum_rgb / total_pixels
    std_rgb = np.sqrt(sum_sq / total_pixels - mean_rgb**2)
    return mean_rgb, std_rgb

MEAN, STD = compute_rgb_stats(train_val['image_path'])
print(f"Image RGB Mean: {MEAN}, Std: {STD}")

# === Phase 12: Define tf.data Pipelines ===
def parse_csv_line(line):
    parts = tf.strings.split(line, sep=",")
    path = parts[0]
    label = tf.strings.to_number(parts[1], tf.float32)
    return path, label

def preprocess_image(path, label, augment=False):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)

    h, w = tf.shape(img)[0], tf.shape(img)[1]
    frac = tf.cast(tf.minimum(h, w), tf.float32) / tf.cast(tf.maximum(h, w), tf.float32)
    img = tf.image.central_crop(img, central_fraction=frac)

    img = tf.image.resize(img, [CONFIG['INPUT_SIZE'], CONFIG['INPUT_SIZE']])

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)

        zoom_factor = tf.random.uniform([], 0.9, 1.1)
        crop_size = tf.cast(zoom_factor * tf.cast(tf.shape(img)[:2], tf.float32), tf.int32)
        img = tf.image.resize_with_crop_or_pad(img, crop_size[0], crop_size[1])
        img = tf.image.resize(img, [CONFIG['INPUT_SIZE'], CONFIG['INPUT_SIZE']])

        k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
        img = tf.image.rot90(img, k)

    img = (img - MEAN) / STD
    return img, label

def make_dataset(csv_file, augment=False, shuffle=False):
    ds = tf.data.TextLineDataset([csv_file]) \
        .skip(1) \
        .map(parse_csv_line, num_parallel_calls=tf.data.AUTOTUNE) \
        .map(lambda path, age: preprocess_image(path, age, augment), num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1024, seed=CONFIG['SEED'])
    return ds.batch(CONFIG['BATCH_SIZE']).prefetch(tf.data.AUTOTUNE)

Unzipped train_images.zip successfully.
Missing files: 0
Corrupted files: 0

=== Resolution BEFORE Rotation ===
   width  height  count
0    480     640  10403
1    640     480      4
Rotated 4 landscape images.

=== Resolution AFTER Rotation ===
   width  height  count
0    480     640  10407
Image RGB Mean: [0.49692728 0.58833559 0.22903944], Std: [0.24089137 0.24288366 0.2172656 ]


# Modeling

In [3]:
# === Imports (assumed already available globally) ===
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras import layers, models, optimizers, callbacks

## 1. Predefined Function

### Plotting

In [4]:
def plot_training_history(model_name):
    history_csv_path = os.path.join(CONFIG['HISTORY_DIR'], model_name, "history.csv")
    history = pd.read_csv(history_csv_path)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
    ax1.plot(history['loss'], label='train loss')
    ax1.plot(history['val_loss'], label='val loss')
    ax1.set_title(f'{model_name} - Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('MSE Loss')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history['mean_absolute_error'], label='train MAE')
    ax2.plot(history['val_mean_absolute_error'], label='val MAE')
    ax2.set_title(f'{model_name} - MAE')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('MAE')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['HISTORY_DIR'], model_name, "training_plot.png"))
    plt.close()

In [5]:
def plot_predicted_vs_actual(true_age, predicted_age, model_name):
    plt.figure(figsize=(6,6))
    plt.scatter(true_age, predicted_age, alpha=0.5)
    min_val = min(true_age.min(), predicted_age.min())
    max_val = max(true_age.max(), predicted_age.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    plt.xlabel("True Age (days)")
    plt.ylabel("Predicted Age (days)")
    plt.title(f"{model_name} - Predicted vs Actual")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['HISTORY_DIR'], model_name, "predicted_vs_actual.png"))
    plt.close()

In [6]:
def plot_residuals(true_age, predicted_age, model_name):
    residuals = true_age - predicted_age
    plt.figure(figsize=(6,5))
    plt.scatter(true_age, residuals, alpha=0.5)
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel("True Age (days)")
    plt.ylabel("Residual (True - Predicted)")
    plt.title(f"{model_name} - Residual Plot")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['HISTORY_DIR'], model_name, "residuals.png"))
    plt.close()

### Training

In [7]:
def train_and_evaluate_model(model, model_name, age_mean, age_std):
    train_ds = make_dataset(CONFIG['TRAIN_CSV'], augment=True, shuffle=True)
    val_ds = make_dataset(CONFIG['TEST_CSV'], augment=False)
    test_ds = make_dataset(CONFIG['TEST_CSV'], augment=False)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['EARLY_STOP_PATIENCE'],
        restore_best_weights=True
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CONFIG['EPOCHS'],
        callbacks=[early_stop],
        verbose=1
    )

    os.makedirs(os.path.join(CONFIG['HISTORY_DIR'], model_name), exist_ok=True)
    model.save(os.path.join(CONFIG['MODEL_DIR'], f"{model_name}.h5"))
    pd.DataFrame(history.history).to_csv(
        os.path.join(CONFIG['HISTORY_DIR'], model_name, "history.csv"), index=False
    )

    test_df = pd.read_csv(CONFIG['TEST_CSV'])
    true_scaled = test_df["age_scaled"].values
    true_unscaled = true_scaled * age_std + age_mean

    y_pred_scaled = model.predict(test_ds).flatten()
    y_pred_unscaled = y_pred_scaled * age_std + age_mean

    mae = mean_absolute_error(true_unscaled, y_pred_unscaled)
    mse = mean_squared_error(true_unscaled, y_pred_unscaled)
    r2 = r2_score(true_unscaled, y_pred_unscaled)

    print(f"✅ {model_name} - MAE: {mae:.2f}, MSE: {mse:.2f}, R²: {r2:.3f}")

    plot_training_history(model_name)
    plot_predicted_vs_actual(true_unscaled, y_pred_unscaled, model_name)
    plot_residuals(true_unscaled, y_pred_unscaled, model_name)

    return mae, mse, r2

## 2. Resnet50

In [8]:
def resnet50(input_shape=(224, 224, 3)):
    def conv_block(x, filters, kernel_size=3, strides=1):
        x = layers.Conv2D(filters, kernel_size, strides=strides, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        return x

    def identity_block(x, filters):
        shortcut = x
        x = conv_block(x, filters)
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Add()([x, shortcut])
        x = layers.ReLU()(x)
        return x

    def projection_block(x, filters, strides):
        shortcut = layers.Conv2D(filters, 1, strides=strides, padding='same', use_bias=False)(x)
        shortcut = layers.BatchNormalization()(shortcut)

        x = conv_block(x, filters, strides=strides)
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)

        x = layers.Add()([x, shortcut])
        x = layers.ReLU()(x)
        return x

    inputs = layers.Input(shape=input_shape)
    x = conv_block(inputs, 64, kernel_size=7, strides=2)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    filter_sizes = [64, 128, 256, 512]
    repetitions = [3, 4, 6, 3]

    for filters, reps in zip(filter_sizes, repetitions):
        x = projection_block(x, filters, strides=2)
        for _ in range(reps - 1):
            x = identity_block(x, filters)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1)(x)

    model = models.Model(inputs=inputs, outputs=outputs)

    steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=CONFIG['LEARNING_RATE'],
        decay_steps=5 * steps_per_epoch,
        decay_rate=0.5,
        staircase=True
    )
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    model.compile(optimizer=optimizer, loss='mse', metrics=[tf.keras.metrics.MeanAbsoluteError()])
    return model

In [9]:
model1 = resnet50()
train_and_evaluate_model(model1, "ResNet50", age_mean, age_std)

Epoch 1/100
261/261 [==============================] - 58s 166ms/step - loss: 1.2288 - mean_absolute_error: 0.8279 - val_loss: 1.0715 - val_mean_absolute_error: 0.8877
Epoch 2/100
261/261 [==============================] - 38s 138ms/step - loss: 0.8613 - mean_absolute_error: 0.7581 - val_loss: 0.7950 - val_mean_absolute_error: 0.7152
Epoch 3/100
261/261 [==============================] - 38s 138ms/step - loss: 0.8558 - mean_absolute_error: 0.7563 - val_loss: 0.8087 - val_mean_absolute_error: 0.7291
Epoch 4/100
261/261 [==============================] - 40s 145ms/step - loss: 0.8223 - mean_absolute_error: 0.7380 - val_loss: 0.8362 - val_mean_absolute_error: 0.7036
Epoch 5/100
261/261 [==============================] - 41s 147ms/step - loss: 0.8011 - mean_absolute_error: 0.7249 - val_loss: 0.7799 - val_mean_absolute_error: 0.7128
Epoch 6/100
261/261 [==============================] - 40s 146ms/step - loss: 0.7526 - mean_absolute_error: 0.6963 - val_loss: 0.6746 - val_mean_absolute_error:

(3.3199007417237265, 23.469345978166807, 0.7073238909820232)

## Resnet50 - Full version

In [10]:
def resnet50Full(input_shape=(224, 224, 3)):
    def bottleneck_block(x, filters, strides=1, downsample=False):
        shortcut = x

        # First 1x1 conv (reduce dim)
        x = layers.Conv2D(filters, kernel_size=1, strides=strides, use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        # 3x3 conv
        x = layers.Conv2D(filters, kernel_size=3, strides=1, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        # Second 1x1 conv (expand to 4× filters)
        x = layers.Conv2D(filters * 4, kernel_size=1, strides=1, use_bias=False)(x)
        x = layers.BatchNormalization()(x)

        # Projection shortcut if needed (match dims)
        if downsample or shortcut.shape[-1] != filters * 4:
            shortcut = layers.Conv2D(filters * 4, kernel_size=1, strides=strides, use_bias=False)(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)

        # Add residual connection
        x = layers.Add()([x, shortcut])
        x = layers.ReLU()(x)
        return x

    def build_resnet_stage(x, filters, blocks, first_stride):
        x = bottleneck_block(x, filters, strides=first_stride, downsample=True)
        for _ in range(1, blocks):
            x = bottleneck_block(x, filters, strides=1)
        return x

    inputs = layers.Input(shape=input_shape)

    # Initial conv and maxpool
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    # Stage-wise blocks [3, 4, 6, 3]
    x = build_resnet_stage(x, filters=64,  blocks=3, first_stride=1)  # conv2_x
    x = build_resnet_stage(x, filters=128, blocks=4, first_stride=2)  # conv3_x
    x = build_resnet_stage(x, filters=256, blocks=6, first_stride=2)  # conv4_x
    x = build_resnet_stage(x, filters=512, blocks=3, first_stride=2)  # conv5_x

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1)(x)

    model = models.Model(inputs=inputs, outputs=outputs)

    # Learning rate schedule
    steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=CONFIG['LEARNING_RATE'],
        decay_steps=5 * steps_per_epoch,
        decay_rate=0.5,
        staircase=True
    )
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    model.compile(optimizer=optimizer, loss='mse', metrics=[tf.keras.metrics.MeanAbsoluteError()])
    return model

In [11]:
model2 = resnet50Full()
train_and_evaluate_model(model2, "ResNet50Full", age_mean, age_std)

Epoch 1/100
261/261 [==============================] - 116s 405ms/step - loss: 1.9448 - mean_absolute_error: 0.9161 - val_loss: 0.9892 - val_mean_absolute_error: 0.8760
Epoch 2/100
261/261 [==============================] - 119s 443ms/step - loss: 0.8948 - mean_absolute_error: 0.7792 - val_loss: 0.8642 - val_mean_absolute_error: 0.8003
Epoch 3/100
261/261 [==============================] - 118s 443ms/step - loss: 0.8613 - mean_absolute_error: 0.7603 - val_loss: 0.8371 - val_mean_absolute_error: 0.7685
Epoch 4/100
261/261 [==============================] - 109s 406ms/step - loss: 0.8428 - mean_absolute_error: 0.7534 - val_loss: 0.7453 - val_mean_absolute_error: 0.6871
Epoch 5/100
261/261 [==============================] - 114s 428ms/step - loss: 0.8140 - mean_absolute_error: 0.7326 - val_loss: 0.7741 - val_mean_absolute_error: 0.6889
Epoch 6/100
261/261 [==============================] - 114s 427ms/step - loss: 0.7599 - mean_absolute_error: 0.7043 - val_loss: 0.7523 - val_mean_absolute_

(3.743985600247049, 27.390165555803495, 0.6584290381296367)

## MobileNetV2

In [28]:
def mobilenetv2(input_shape=(224, 224, 3)):
    def relu6(x):
        return tf.keras.activations.relu(x, max_value=6.0)

    def inverted_res_block(x, expand, out_channels, stride, block_id):
        in_channels = x.shape[-1]
        residual = x  # Preserve original input for residual connection

        # 1. Expansion
        if expand != 1:
            x = layers.Conv2D(in_channels * expand, 1, padding='same', use_bias=False, name=f"expand_{block_id}")(x)
            x = layers.BatchNormalization(name=f"expand_bn_{block_id}")(x)
            x = layers.Activation(relu6, name=f"expand_relu_{block_id}")(x)

        # 2. Depthwise
        x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False, name=f"dwconv_{block_id}")(x)
        x = layers.BatchNormalization(name=f"dw_bn_{block_id}")(x)
        x = layers.Activation(relu6, name=f"dw_relu_{block_id}")(x)

        # 3. Projection
        x = layers.Conv2D(out_channels, 1, padding='same', use_bias=False, name=f"project_{block_id}")(x)
        x = layers.BatchNormalization(name=f"project_bn_{block_id}")(x)

        # 4. Residual connection
        if stride == 1 and in_channels == out_channels:
            x = layers.Add(name=f"add_{block_id}")([x, residual])

        return x

    inputs = layers.Input(shape=input_shape)

    # Initial conv
    x = layers.Conv2D(32, kernel_size=3, strides=2, padding='same', use_bias=False, name="initial_conv")(inputs)
    x = layers.BatchNormalization(name="initial_bn")(x)
    x = layers.Activation(relu6, name="initial_relu")(x)

    # MobileNetV2 block config: (t, c, n, s)
    config = [
        (1,  16, 1, 1),
        (6,  24, 2, 2),
        (6,  32, 3, 2),
        (6,  64, 4, 2),
        (6,  96, 3, 1),
        (6, 160, 3, 2),
        (6, 320, 1, 1),
    ]

    block_id = 0
    for t, c, n, s in config:
        for i in range(n):
            stride = s if i == 0 else 1
            x = inverted_res_block(x, expand=t, out_channels=c, stride=stride, block_id=block_id)
            block_id += 1

    # Final layers
    x = layers.Conv2D(1280, kernel_size=1, use_bias=False, name="final_conv")(x)
    x = layers.BatchNormalization(name="final_bn")(x)
    x = layers.Activation(relu6, name="final_relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1)(x)

    model = models.Model(inputs, output, name="MobileNetV2_Scratch")

    # Learning rate schedule
    steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=CONFIG['LEARNING_RATE'],
        decay_steps=5 * steps_per_epoch,
        decay_rate=0.5,
        staircase=True
    )

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
    model.compile(optimizer=optimizer, loss='mse', metrics=[tf.keras.metrics.MeanAbsoluteError()])
    return model

In [29]:
model3 = mobilenetv2()
train_and_evaluate_model(model3, "MobileNetV2", age_mean, age_std)

Epoch 1/100
261/261 [==============================] - 66s 198ms/step - loss: 1.0206 - mean_absolute_error: 0.8292 - val_loss: 1.2644 - val_mean_absolute_error: 0.9784
Epoch 2/100
261/261 [==============================] - 51s 177ms/step - loss: 0.8758 - mean_absolute_error: 0.7709 - val_loss: 1.0655 - val_mean_absolute_error: 0.9065
Epoch 3/100
261/261 [==============================] - 49s 173ms/step - loss: 0.8423 - mean_absolute_error: 0.7499 - val_loss: 1.0630 - val_mean_absolute_error: 0.7616
Epoch 4/100
261/261 [==============================] - 50s 175ms/step - loss: 0.8249 - mean_absolute_error: 0.7402 - val_loss: 1.2947 - val_mean_absolute_error: 0.8568
Epoch 5/100
261/261 [==============================] - 51s 179ms/step - loss: 0.7846 - mean_absolute_error: 0.7158 - val_loss: 1.2427 - val_mean_absolute_error: 0.8410
Epoch 6/100
261/261 [==============================] - 52s 185ms/step - loss: 0.7409 - mean_absolute_error: 0.6895 - val_loss: 0.7808 - val_mean_absolute_error:

(4.780696526269977, 41.02764245068162, 0.4883619353595554)

## EfficientNetB0

In [ ]:
# def efficientnetb0(input_shape=(224, 224, 3)):
#     def swish(x):
#         return x * tf.nn.sigmoid(x)

#     def se_block(input_tensor, reduction=4):
#         filters = input_tensor.shape[-1]
#         se = layers.GlobalAveragePooling2D()(input_tensor)
#         se = layers.Reshape((1, 1, filters))(se)
#         se = layers.Dense(filters // reduction, activation='swish', use_bias=False)(se)
#         se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
#         return layers.multiply([input_tensor, se])

#     def mbconv_block(x, in_channels, out_channels, kernel_size, strides, expand_ratio, se_ratio=0.25, block_id=0):
#         shortcut = x
#         expanded = in_channels * expand_ratio

#         # 1. Expansion (1x1)
#         if expand_ratio != 1:
#             x = layers.Conv2D(expanded, 1, padding='same', use_bias=False, name=f"expand_{block_id}")(x)
#             x = layers.BatchNormalization(name=f"expand_bn_{block_id}")(x)
#             x = layers.Activation(swish, name=f"expand_act_{block_id}")(x)

#         # 2. Depthwise conv
#         x = layers.DepthwiseConv2D(kernel_size, strides=strides, padding='same', use_bias=False, name=f"dwconv_{block_id}")(x)
#         x = layers.BatchNormalization(name=f"dw_bn_{block_id}")(x)
#         x = layers.Activation(swish, name=f"dw_act_{block_id}")(x)

#         # 3. Squeeze-and-Excitation
#         x = se_block(x)

#         # 4. Projection (1x1)
#         x = layers.Conv2D(out_channels, 1, padding='same', use_bias=False, name=f"project_{block_id}")(x)
#         x = layers.BatchNormalization(name=f"project_bn_{block_id}")(x)

#         # 5. Skip connection if possible
#         if strides == 1 and in_channels == out_channels:
#             x = layers.Add(name=f"skip_{block_id}")([x, shortcut])

#         return x

#     inputs = layers.Input(shape=input_shape)

#     # Stem
#     x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False, name="stem_conv")(inputs)
#     x = layers.BatchNormalization(name="stem_bn")(x)
#     x = layers.Activation(swish, name="stem_swish")(x)

#     # Block configuration: (repeats, kernel, strides, expand_ratio, in_ch, out_ch)
#     blocks = [
#         (1, 3, 1, 1,   32,  16),
#         (2, 3, 2, 6,   16,  24),
#         (2, 5, 2, 6,   24,  40),
#         (3, 3, 2, 6,   40,  80),
#         (3, 5, 1, 6,   80, 112),
#         (4, 5, 2, 6,  112, 192),
#         (1, 3, 1, 6,  192, 320),
#     ]

#     block_id = 0
#     for repeats, kernel, stride, expand_ratio, in_ch, out_ch in blocks:
#         for i in range(repeats):
#             s = stride if i == 0 else 1
#             x = mbconv_block(x, in_ch if i == 0 else out_ch, out_ch, kernel, s, expand_ratio, block_id=block_id)
#             block_id += 1

#     # Head
#     x = layers.Conv2D(1280, 1, padding='same', use_bias=False, name="head_conv")(x)
#     x = layers.BatchNormalization(name="head_bn")(x)
#     x = layers.Activation(swish, name="head_swish")(x)

#     x = layers.GlobalAveragePooling2D()(x)
#     x = layers.Dense(128, activation='relu')(x)
#     x = layers.Dropout(0.3)(x)
#     outputs = layers.Dense(1)(x)

#     model = models.Model(inputs=inputs, outputs=outputs, name="EfficientNetB0_Scratch")

#     # Learning rate schedule
#     steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
#     lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
#         initial_learning_rate=CONFIG['LEARNING_RATE'],
#         decay_steps=5 * steps_per_epoch,
#         decay_rate=0.5,
#         staircase=True
#     )
#     optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

#     model.compile(optimizer=optimizer, loss='mse', metrics=[tf.keras.metrics.MeanAbsoluteError()])
#     return model

In [30]:
def efficientnetb0(input_shape=(224, 224, 3)):
    def swish(x):
        return x * tf.keras.backend.sigmoid(x)

    def conv_bn_swish(x, filters, kernel_size, stride, name):
        x = layers.Conv2D(filters, kernel_size, strides=stride, padding='same', use_bias=False, name=f"{name}_conv")(x)
        x = layers.BatchNormalization(name=f"{name}_bn")(x)
        x = layers.Activation(swish, name=f"{name}_swish")(x)
        return x

    def mbconv_block(x, in_channels, out_channels, expand_ratio, stride, block_id):
        hidden_dim = in_channels * expand_ratio
        prefix = f"block_{block_id}"

        # Expand
        if expand_ratio != 1:
            x = layers.Conv2D(hidden_dim, 1, padding='same', use_bias=False, name=f"{prefix}_expand_conv")(x)
            x = layers.BatchNormalization(name=f"{prefix}_expand_bn")(x)
            x = layers.Activation(swish, name=f"{prefix}_expand_swish")(x)

        # Depthwise
        x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False, name=f"{prefix}_dwconv")(x)
        x = layers.BatchNormalization(name=f"{prefix}_dw_bn")(x)
        x = layers.Activation(swish, name=f"{prefix}_dw_swish")(x)

        # Project
        x = layers.Conv2D(out_channels, 1, padding='same', use_bias=False, name=f"{prefix}_project_conv")(x)
        x = layers.BatchNormalization(name=f"{prefix}_project_bn")(x)

        # Residual
        if stride == 1 and in_channels == out_channels:
            x = layers.Add(name=f"{prefix}_add")([x, input_tensor])
        return x

    inputs = layers.Input(shape=input_shape)
    x = conv_bn_swish(inputs, 32, 3, 2, "stem")

    # Simplified MBConv block config: (expand_ratio, out_channels, repeats, stride)
    config = [
        (1, 16, 1, 1),
        (6, 24, 2, 2),
        (6, 40, 2, 2),
        (6, 80, 3, 2),
        (6, 112, 3, 1),
        (6, 192, 4, 2),
        (6, 320, 1, 1),
    ]

    block_id = 0
    for expand_ratio, out_channels, repeats, stride in config:
        for i in range(repeats):
            s = stride if i == 0 else 1
            input_tensor = x
            x = mbconv_block(x, x.shape[-1], out_channels, expand_ratio, s, block_id)
            block_id += 1

    x = conv_bn_swish(x, 1280, 1, 1, "head")
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1)(x)

    model = models.Model(inputs, outputs, name="EfficientNetB0_Scratch")

    # Learning rate decay schedule
    steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=CONFIG['LEARNING_RATE'],
        decay_steps=5 * steps_per_epoch,
        decay_rate=0.5,
        staircase=True
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss='mse',
        metrics=[tf.keras.metrics.MeanAbsoluteError()]
    )
    return model

In [31]:
model4 = efficientnetb0()
train_and_evaluate_model(model4, "EfficientNetB0", age_mean, age_std)

Epoch 1/100


ResourceExhaustedError: Graph execution error:

Detected at node 'EfficientNetB0_Scratch/block_8_expand_conv/Conv2D' defined at (most recent call last):
    File "c:\Users\Admin\.conda\envs\py310\lib\runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "c:\Users\Admin\.conda\envs\py310\lib\runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "c:\Users\Admin\.conda\envs\py310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "c:\Users\Admin\.conda\envs\py310\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "c:\Users\Admin\.conda\envs\py310\lib\site-packages\ipykernel\kernelapp.py", line 739, in start
      self.io_loop.start()
    File "c:\Users\Admin\.conda\envs\py310\lib\site-packages\tornado\platform\asyncio.py", line 205, in start
      self.asyncio_loop.run_forever()
    File "c:\Users\Admin\.conda\envs

## DenseNet

In [ ]:
# def densenet121(input_shape=(224, 224, 3), growth_rate=32, compression=0.5):
#     def conv_block(x, growth_rate, block_name):
#         """Bottleneck conv block: BN -> ReLU -> 1x1 -> BN -> ReLU -> 3x3"""
#         x1 = layers.BatchNormalization(name=f"{block_name}_bn1")(x)
#         x1 = layers.ReLU(name=f"{block_name}_relu1")(x1)
#         x1 = layers.Conv2D(4 * growth_rate, 1, use_bias=False, name=f"{block_name}_conv1")(x1)

#         x1 = layers.BatchNormalization(name=f"{block_name}_bn2")(x1)
#         x1 = layers.ReLU(name=f"{block_name}_relu2")(x1)
#         x1 = layers.Conv2D(growth_rate, 3, padding='same', use_bias=False, name=f"{block_name}_conv2")(x1)

#         x = layers.Concatenate(name=f"{block_name}_concat")([x, x1])
#         return x

#     def dense_block(x, num_layers, growth_rate, block_idx):
#         for i in range(num_layers):
#             x = conv_block(x, growth_rate, block_name=f"db{block_idx}_layer{i+1}")
#         return x

#     def transition_layer(x, compression, block_idx):
#         filters = int(x.shape[-1] * compression)
#         x = layers.BatchNormalization(name=f"trans{block_idx}_bn")(x)
#         x = layers.ReLU(name=f"trans{block_idx}_relu")(x)
#         x = layers.Conv2D(filters, 1, use_bias=False, name=f"trans{block_idx}_conv")(x)
#         x = layers.AveragePooling2D(2, strides=2, name=f"trans{block_idx}_pool")(x)
#         return x

#     inputs = layers.Input(shape=input_shape)

#     # Initial Conv
#     x = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False, name="init_conv")(inputs)
#     x = layers.BatchNormalization(name="init_bn")(x)
#     x = layers.ReLU(name="init_relu")(x)
#     x = layers.MaxPooling2D(3, strides=2, padding='same', name="init_pool")(x)

#     # Dense Blocks + Transitions
#     block_config = [6, 12, 24, 16]
#     for i, num_layers in enumerate(block_config):
#         x = dense_block(x, num_layers, growth_rate, block_idx=i+1)
#         if i != len(block_config) - 1:
#             x = transition_layer(x, compression, block_idx=i+1)

#     # Final layers
#     x = layers.BatchNormalization(name="final_bn")(x)
#     x = layers.ReLU(name="final_relu")(x)
#     x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)

#     # Regression head
#     x = layers.Dense(128, activation='relu')(x)
#     x = layers.Dropout(0.3)(x)
#     outputs = layers.Dense(1)(x)

#     model = models.Model(inputs, outputs, name="DenseNet121_Scratch")

#     # Learning rate schedule
#     steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
#     lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
#         initial_learning_rate=CONFIG['LEARNING_RATE'],
#         decay_steps=5 * steps_per_epoch,
#         decay_rate=0.5,
#         staircase=True
#     )

#     model.compile(
#         optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
#         loss='mse',
#         metrics=[tf.keras.metrics.MeanAbsoluteError()]
#     )
#     return model

In [26]:
def densenet121(input_shape=(224, 224, 3)):
    def conv_block(x, growth_rate, block_id):
        """A single convolution block inside a dense block."""
        x1 = layers.BatchNormalization(name=f"bn_{block_id}")(x)
        x1 = layers.Activation('relu', name=f"relu_{block_id}")(x1)
        x1 = layers.Conv2D(growth_rate, 3, padding='same', name=f"conv_{block_id}")(x1)
        x = layers.Concatenate(name=f"concat_{block_id}")([x, x1])
        return x

    def dense_block(x, num_layers, growth_rate, name):
        for i in range(num_layers):
            x = conv_block(x, growth_rate, f"{name}_layer{i+1}")
        return x

    def transition_layer(x, reduction, name):
        x = layers.BatchNormalization(name=f"{name}_bn")(x)
        x = layers.Activation('relu', name=f"{name}_relu")(x)
        x = layers.Conv2D(int(x.shape[-1] * reduction), 1, padding='same', name=f"{name}_conv")(x)
        x = layers.AveragePooling2D(2, strides=2, name=f"{name}_pool")(x)
        return x

    inputs = layers.Input(shape=input_shape)

    # Initial Conv
    x = layers.Conv2D(64, 7, strides=2, padding='same', name="initial_conv")(inputs)
    x = layers.BatchNormalization(name="initial_bn")(x)
    x = layers.Activation('relu', name="initial_relu")(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same', name="initial_pool")(x)

    # Dense Blocks with Transition Layers
    x = dense_block(x, num_layers=6, growth_rate=32, name="dense1")
    x = transition_layer(x, reduction=0.5, name="trans1")

    x = dense_block(x, num_layers=12, growth_rate=32, name="dense2")
    x = transition_layer(x, reduction=0.5, name="trans2")

    x = dense_block(x, num_layers=24, growth_rate=32, name="dense3")
    x = transition_layer(x, reduction=0.5, name="trans3")

    x = dense_block(x, num_layers=16, growth_rate=32, name="dense4")

    x = layers.BatchNormalization(name="final_bn")(x)
    x = layers.Activation('relu', name="final_relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1)(x)

    model = models.Model(inputs, outputs, name="DenseNet121_Scratch")

    # Learning rate schedule
    steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=CONFIG['LEARNING_RATE'],
        decay_steps=5 * steps_per_epoch,
        decay_rate=0.5,
        staircase=True
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss='mse',
        metrics=[tf.keras.metrics.MeanAbsoluteError()]
    )
    return model

In [27]:
model6 = densenet121()
train_and_evaluate_model(model6, "DenseNet121", age_mean, age_std)

Epoch 1/100


NotFoundError: Graph execution error:

Detected at node 'DenseNet121_Scratch/conv_dense3_layer8/Conv2D' defined at (most recent call last):
    File "c:\Users\Admin\.conda\envs\py310\lib\runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "c:\Users\Admin\.conda\envs\py310\lib\runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "c:\Users\Admin\.conda\envs\py310\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "c:\Users\Admin\.conda\envs\py310\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "c:\Users\Admin\.conda\envs\py310\lib\site-packages\ipykernel\kernelapp.py", line 739, in start
      self.io_loop.start()
    File "c:\Users\Admin\.conda\envs\py310\lib\site-packages\tornado\platform\asyncio.py", line 205, in start
      self.asyncio_loop.run_forever()
    File "c:\Users\Admin\.conda\envs\py310\lib\as

## InceptionV3 - Simplified

In [18]:
def inceptionv3(input_shape=(224, 224, 3)):
    def conv_bn_relu(x, filters, kernel_size, strides=1, padding='same', name=None):
        x = layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, use_bias=False, name=f"{name}_conv")(x)
        x = layers.BatchNormalization(name=f"{name}_bn")(x)
        x = layers.ReLU(name=f"{name}_relu")(x)
        return x

    def inception_a(x, name):
        # 1x1
        path1 = conv_bn_relu(x, 64, 1, name=f"{name}_p1")

        # 3x3
        path2 = conv_bn_relu(x, 48, 1, name=f"{name}_p2a")
        path2 = conv_bn_relu(path2, 64, 3, name=f"{name}_p2b")

        # 3x3 -> 3x3
        path3 = conv_bn_relu(x, 64, 1, name=f"{name}_p3a")
        path3 = conv_bn_relu(path3, 96, 3, name=f"{name}_p3b")
        path3 = conv_bn_relu(path3, 96, 3, name=f"{name}_p3c")

        # AvgPool + 1x1
        path4 = layers.AveragePooling2D(3, strides=1, padding='same', name=f"{name}_p4pool")(x)
        path4 = conv_bn_relu(path4, 32, 1, name=f"{name}_p4")

        return layers.Concatenate(name=f"{name}_concat")([path1, path2, path3, path4])

    def reduction_a(x, name):
        path1 = conv_bn_relu(x, 384, 3, strides=2, padding='valid', name=f"{name}_p1")
        path2 = conv_bn_relu(x, 64, 1, name=f"{name}_p2a")
        path2 = conv_bn_relu(path2, 96, 3, name=f"{name}_p2b")
        path2 = conv_bn_relu(path2, 96, 3, strides=2, padding='valid', name=f"{name}_p2c")
        path3 = layers.MaxPooling2D(3, strides=2, padding='valid', name=f"{name}_p3")(x)
        return layers.Concatenate(name=f"{name}_concat")([path1, path2, path3])

    def inception_b(x, name):
        # 1x1
        path1 = conv_bn_relu(x, 192, 1, name=f"{name}_p1")

        # 1x7 -> 7x1
        path2 = conv_bn_relu(x, 128, 1, name=f"{name}_p2a")
        path2 = conv_bn_relu(path2, 128, (1, 7), name=f"{name}_p2b")
        path2 = conv_bn_relu(path2, 192, (7, 1), name=f"{name}_p2c")

        # Stack of 1x7 + 7x1
        path3 = conv_bn_relu(x, 128, 1, name=f"{name}_p3a")
        path3 = conv_bn_relu(path3, 128, (7, 1), name=f"{name}_p3b")
        path3 = conv_bn_relu(path3, 128, (1, 7), name=f"{name}_p3c")
        path3 = conv_bn_relu(path3, 128, (7, 1), name=f"{name}_p3d")
        path3 = conv_bn_relu(path3, 192, (1, 7), name=f"{name}_p3e")

        # AvgPool + 1x1
        path4 = layers.AveragePooling2D(3, strides=1, padding='same', name=f"{name}_p4pool")(x)
        path4 = conv_bn_relu(path4, 192, 1, name=f"{name}_p4")

        return layers.Concatenate(name=f"{name}_concat")([path1, path2, path3, path4])

    def reduction_b(x, name):
        path1 = conv_bn_relu(x, 192, 1, name=f"{name}_p1a")
        path1 = conv_bn_relu(path1, 320, 3, strides=2, padding='valid', name=f"{name}_p1b")

        path2 = conv_bn_relu(x, 192, 1, name=f"{name}_p2a")
        path2 = conv_bn_relu(path2, 192, (1, 7), name=f"{name}_p2b")
        path2 = conv_bn_relu(path2, 192, (7, 1), name=f"{name}_p2c")
        path2 = conv_bn_relu(path2, 192, 3, strides=2, padding='valid', name=f"{name}_p2d")

        path3 = layers.MaxPooling2D(3, strides=2, padding='valid', name=f"{name}_p3")(x)
        return layers.Concatenate(name=f"{name}_concat")([path1, path2, path3])

    def inception_c(x, name):
        # 1x1
        path1 = conv_bn_relu(x, 320, 1, name=f"{name}_p1")

        # 1x3 + 3x1
        path2 = conv_bn_relu(x, 384, 1, name=f"{name}_p2a")
        path2a = conv_bn_relu(path2, 384, (1, 3), name=f"{name}_p2b1")
        path2b = conv_bn_relu(path2, 384, (3, 1), name=f"{name}_p2b2")
        path2 = layers.Concatenate(name=f"{name}_p2_concat")([path2a, path2b])

        # 3x3 + (1x3 + 3x1)
        path3 = conv_bn_relu(x, 448, 1, name=f"{name}_p3a")
        path3 = conv_bn_relu(path3, 384, 3, name=f"{name}_p3b")
        path3a = conv_bn_relu(path3, 384, (1, 3), name=f"{name}_p3c1")
        path3b = conv_bn_relu(path3, 384, (3, 1), name=f"{name}_p3c2")
        path3 = layers.Concatenate(name=f"{name}_p3_concat")([path3a, path3b])

        # Pool
        path4 = layers.AveragePooling2D(3, strides=1, padding='same', name=f"{name}_p4pool")(x)
        path4 = conv_bn_relu(path4, 192, 1, name=f"{name}_p4")

        return layers.Concatenate(name=f"{name}_concat")([path1, path2, path3, path4])

    inputs = layers.Input(shape=input_shape)

    # Stem
    x = conv_bn_relu(inputs, 32, 3, strides=2, padding='valid', name='stem1')
    x = conv_bn_relu(x, 32, 3, padding='valid', name='stem2')
    x = conv_bn_relu(x, 64, 3, name='stem3')
    x = layers.MaxPooling2D(3, strides=2, padding='valid', name='stem4')(x)
    x = conv_bn_relu(x, 80, 1, name='stem5')
    x = conv_bn_relu(x, 192, 3, padding='valid', name='stem6')
    x = layers.MaxPooling2D(3, strides=2, padding='valid', name='stem7')(x)

    # Inception blocks
    x = inception_a(x, name="inceptA1")
    x = inception_a(x, name="inceptA2")
    x = inception_a(x, name="inceptA3")

    x = reduction_a(x, name="reduceA")

    x = inception_b(x, name="inceptB1")
    x = inception_b(x, name="inceptB2")
    x = inception_b(x, name="inceptB3")
    x = inception_b(x, name="inceptB4")

    x = reduction_b(x, name="reduceB")

    x = inception_c(x, name="inceptC1")
    x = inception_c(x, name="inceptC2")
    x = inception_c(x, name="inceptC3")

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1)(x)

    model = models.Model(inputs, outputs, name="InceptionV3_Scratch")

    # Learning rate schedule
    steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=CONFIG['LEARNING_RATE'],
        decay_steps=5 * steps_per_epoch,
        decay_rate=0.5,
        staircase=True
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss='mse',
        metrics=[tf.keras.metrics.MeanAbsoluteError()]
    )
    return model

In [19]:
model7 = inceptionv3()
train_and_evaluate_model(model7, "InceptionV3", age_mean, age_std)

Epoch 1/100
261/261 [==============================] - 112s 362ms/step - loss: 1.2993 - mean_absolute_error: 0.8295 - val_loss: 1.1544 - val_mean_absolute_error: 0.9317
Epoch 2/100
261/261 [==============================] - 95s 356ms/step - loss: 0.8573 - mean_absolute_error: 0.7591 - val_loss: 0.8025 - val_mean_absolute_error: 0.7081
Epoch 3/100
261/261 [==============================] - 102s 378ms/step - loss: 0.8286 - mean_absolute_error: 0.7421 - val_loss: 0.8397 - val_mean_absolute_error: 0.7174
Epoch 4/100
261/261 [==============================] - 89s 329ms/step - loss: 0.7990 - mean_absolute_error: 0.7256 - val_loss: 0.7963 - val_mean_absolute_error: 0.7208
Epoch 5/100
261/261 [==============================] - 89s 331ms/step - loss: 0.7689 - mean_absolute_error: 0.7055 - val_loss: 0.6691 - val_mean_absolute_error: 0.6626
Epoch 6/100
261/261 [==============================] - 89s 332ms/step - loss: 0.7097 - mean_absolute_error: 0.6736 - val_loss: 0.6505 - val_mean_absolute_erro

(2.7311392315984575, 18.507775688485275, 0.7691974978713709)

## Custom CNN

In [20]:
def customcnn(input_shape=(224, 224, 3)):
    """Upgraded custom CNN model for fair comparison with deeper architectures."""
    inputs = layers.Input(shape=input_shape)
    x = inputs

    # Conv Block 1
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    # Conv Block 2
    x = layers.Conv2D(64, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    # Conv Block 3
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    # Conv Block 4
    x = layers.Conv2D(256, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    # Conv Block 5
    x = layers.Conv2D(512, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    # Global Pooling and Dense Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1)(x)

    model = models.Model(inputs, outputs, name="CustomCNN")

    # Learning rate decay (same as other models)
    steps_per_epoch = len(pd.read_csv(CONFIG['TRAIN_CSV'])) // CONFIG['BATCH_SIZE']
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=CONFIG['LEARNING_RATE'],
        decay_steps=5 * steps_per_epoch,
        decay_rate=0.5,
        staircase=True
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss='mse',
        metrics=[tf.keras.metrics.MeanAbsoluteError()]
    )
    return model

In [21]:
model8 = customcnn()
train_and_evaluate_model(model8, "CustomCNN", age_mean, age_std)

Epoch 1/100
261/261 [==============================] - 33s 109ms/step - loss: 1.1138 - mean_absolute_error: 0.8354 - val_loss: 0.8857 - val_mean_absolute_error: 0.7554
Epoch 2/100
261/261 [==============================] - 29s 102ms/step - loss: 0.8622 - mean_absolute_error: 0.7644 - val_loss: 0.7902 - val_mean_absolute_error: 0.7310
Epoch 3/100
261/261 [==============================] - 29s 102ms/step - loss: 0.8308 - mean_absolute_error: 0.7462 - val_loss: 1.1274 - val_mean_absolute_error: 0.9232
Epoch 4/100
261/261 [==============================] - 29s 102ms/step - loss: 0.8040 - mean_absolute_error: 0.7308 - val_loss: 0.8350 - val_mean_absolute_error: 0.6920
Epoch 5/100
261/261 [==============================] - 29s 103ms/step - loss: 0.7734 - mean_absolute_error: 0.7135 - val_loss: 0.7516 - val_mean_absolute_error: 0.7090
Epoch 6/100
261/261 [==============================] - 29s 102ms/step - loss: 0.7122 - mean_absolute_error: 0.6804 - val_loss: 0.8620 - val_mean_absolute_error:

(3.7032076059684056, 23.836060855805492, 0.7027507476312821)